## Fase #2: Comprensión de los Datos / Data Understanding
La fase de comprensión de datos comienza con una recopilación inicial de datos y continúa con actividades para familiarizarse con los datos, identificar problemas de calidad de los datos, descubrir los primeros conocimientos sobre los datos o detectar subconjuntos interesantes para formular hipótesis sobre información oculta.
* Establecé una metodología de adquisición de los datos para el negocio.
* Contemplá en la adquisición de datos una alternativa donde debas construir el setde datos.
* Identificá los distintos problemas o desafíos con los datos (datos faltantes, ruidos,etc) que puedan afectar a su calidad.
* Enumerá las distintas fases, tareas generales, tareas específicas e instancias del proceso que incluirás como estrategia para aplicar la metodología

### Fuentes de los datos

### Forma de obtención

### Condiciones de uso y licencias

Adquisición, construcción y calidad ---> VER DONDE PONER ESTO XD

**Fuente principal.** Se descarga manualmente el CSV de actualización diaria del Adjunto IV y se conserva sin editar. Cada ejecución registra nombre, tamaño, SHA-256 y fecha máxima de `fecha_data`; así se puede identificar exactamente la versión utilizada.

**Alternativa construida.** Como extensión, se puede unir el Adjunto IV con la producción no convencional mediante `idpozo` y agregar la producción de los primeros seis meses posteriores a `fecha_fin_fractura`. Ese conjunto permitiría estudiar producción inicial según características de completación. Requiere resolver pozos con múltiples intervenciones, meses faltantes y ventanas incompletas, por lo que no forma parte del modelo principal.

**Desafíos de calidad.** Se controlarán duplicados de `id_base_fractura_adjiv`, valores no numéricos, ceros, fechas inválidas o invertidas, categorías poco frecuentes, pozos repetidos y valores extremos. Los filtros se aplican de manera secuencial para que el resumen de exclusiones sea reproducible. No se imputará la variable objetivo.

In [ ]:
# @title Imports
import pandas as pd
import requests
import urllib3

from comun import RUTA_DATOS


### Descarga de archivo

In [ ]:
# @title Descargar datos de energía
# Silenciar advertencias de certificados no verificados
def descargar_dataset():
	urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

	url = "https://datos.energia.gob.ar/dataset/71fa2e84-0316-4a1b-af68-7f35e41f58d7/resource/2280ad92-6ed3-403e-a095-50139863ab0d/download/datos-de-fractura-de-pozos-de-hidrocarburos-adjunto-iv-actualizacin-diaria.csv"
	headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}

	# Realizar la descarga en bloques (stream) para evitar sobrecargar memoria
	with requests.get(url, headers=headers, verify=False, stream=True, timeout=60) as r:
		r.raise_for_status()
		with open(RUTA_DATOS, "wb") as f:
			for chunk in r.iter_content(chunk_size=8192):
				f.write(chunk)

	print("Descarga completada con éxito.")


descargar_dataset()

### Carga de datos

In [ ]:
# @title Cargar datos

# Cargar el archivo CSV descargado en un DataFrame de pandas
df_datos_energia = pd.read_csv(RUTA_DATOS)

# Visualizar las primeras filas del DataFrame
df_datos_energia.head()

### Exploración inicial

In [ ]:
# Determinar el número de filas y columnas del dataset
num_filas, num_columnas = df_datos_energia.shape

# Mostrar el número de filas y columnas
print(f"El Dataset tiene {num_filas} filas y {num_columnas} columnas.")

In [ ]:
df_datos_energia.info()

In [ ]:
df_datos_energia.isnull().sum()

In [ ]:
# Definición del diccionario con el mapeo de variables --FIIJARSE BIEN ESTO XD---
mapeo_variables = [
	{
		"Variable Original": "id_base_fractura_adjiv",
		"Tipo Actual (Pandas)": str(df_datos_energia["id_base_fractura_adjiv"].dtype),
		"Nombre Propuesto": "id_fractura",
		"Tipo Correspondiente": "Categórico / Identificador",
		"Dtype Sugerido": "string",
		"Valores Nulos": int(df_datos_energia["id_base_fractura_adjiv"].isnull().sum()),
		"Observaciones / Acción": "Clave primaria del registro. No operar matemáticamente.",
	},
	{
		"Variable Original": "idpozo",
		"Tipo Actual (Pandas)": str(df_datos_energia["idpozo"].dtype),
		"Nombre Propuesto": "id_pozo",
		"Tipo Correspondiente": "Categórico / Identificador",
		"Dtype Sugerido": "string",
		"Valores Nulos": int(df_datos_energia["idpozo"].isnull().sum()),
		"Observaciones / Acción": "Identificador único de pozo (foreign key).",
	},
	{
		"Variable Original": "sigla",
		"Tipo Actual (Pandas)": str(df_datos_energia["sigla"].dtype),
		"Nombre Propuesto": "pozo_sigla",
		"Tipo Correspondiente": "Categórico nominal",
		"Dtype Sugerido": "category",
		"Valores Nulos": int(df_datos_energia["sigla"].isnull().sum()),
		"Observaciones / Acción": "Código o nombre identificador del pozo.",
	},
	{
		"Variable Original": "cuenca",
		"Tipo Actual (Pandas)": str(df_datos_energia["cuenca"].dtype),
		"Nombre Propuesto": "cuenca",
		"Tipo Correspondiente": "Categórico nominal",
		"Dtype Sugerido": "category",
		"Valores Nulos": int(df_datos_energia["cuenca"].isnull().sum()),
		"Observaciones / Acción": "Cuenca geológica (baja cardinalidad).",
	},
	{
		"Variable Original": "areapermisoconcesion",
		"Tipo Actual (Pandas)": str(df_datos_energia["areapermisoconcesion"].dtype),
		"Nombre Propuesto": "area_concesion",
		"Tipo Correspondiente": "Categórico nominal",
		"Dtype Sugerido": "category",
		"Valores Nulos": int(df_datos_energia["areapermisoconcesion"].isnull().sum()),
		"Observaciones / Acción": "Bloque o área hidrocarburífera de concesión.",
	},
	{
		"Variable Original": "yacimiento",
		"Tipo Actual (Pandas)": str(df_datos_energia["yacimiento"].dtype),
		"Nombre Propuesto": "yacimiento",
		"Tipo Correspondiente": "Categórico nominal",
		"Dtype Sugerido": "category",
		"Valores Nulos": int(df_datos_energia["yacimiento"].isnull().sum()),
		"Observaciones / Acción": "Yacimiento productivo.",
	},
	{
		"Variable Original": "formacion_productiva",
		"Tipo Actual (Pandas)": str(df_datos_energia["formacion_productiva"].dtype),
		"Nombre Propuesto": "formacion_productiva",
		"Tipo Correspondiente": "Categórico nominal",
		"Dtype Sugerido": "category",
		"Valores Nulos": int(df_datos_energia["formacion_productiva"].isnull().sum()),
		"Observaciones / Acción": "Formación objetivo (ej. Vaca Muerta).",
	},
	{
		"Variable Original": "tipo_reservorio",
		"Tipo Actual (Pandas)": str(df_datos_energia["tipo_reservorio"].dtype),
		"Nombre Propuesto": "tipo_reservorio",
		"Tipo Correspondiente": "Categórico nominal",
		"Dtype Sugerido": "category",
		"Valores Nulos": int(df_datos_energia["tipo_reservorio"].isnull().sum()),
		"Observaciones / Acción": "Convencional / No Convencional. Evaluar imputación de nulos.",
	},
	{
		"Variable Original": "subtipo_reservorio",
		"Tipo Actual (Pandas)": str(df_datos_energia["subtipo_reservorio"].dtype),
		"Nombre Propuesto": "subtipo_reservorio",
		"Tipo Correspondiente": "Categórico nominal",
		"Dtype Sugerido": "category",
		"Valores Nulos": int(df_datos_energia["subtipo_reservorio"].isnull().sum()),
		"Observaciones / Acción": "Shale / Tight. 973 nulos a tratar en Fase 3.",
	},
	{
		"Variable Original": "longitud_rama_horizontal_m",
		"Tipo Actual (Pandas)": str(df_datos_energia["longitud_rama_horizontal_m"].dtype),
		"Nombre Propuesto": "longitud_rama_horizontal_m",
		"Tipo Correspondiente": "Numérico continuo",
		"Dtype Sugerido": "float64",
		"Valores Nulos": int(df_datos_energia["longitud_rama_horizontal_m"].isnull().sum()),
		"Observaciones / Acción": "Longitud perforada en metros.",
	},
	{
		"Variable Original": "cantidad_fracturas",
		"Tipo Actual (Pandas)": str(df_datos_energia["cantidad_fracturas"].dtype),
		"Nombre Propuesto": "etapas_fractura",
		"Tipo Correspondiente": "Numérico discreto",
		"Dtype Sugerido": "int64",
		"Valores Nulos": int(df_datos_energia["cantidad_fracturas"].isnull().sum()),
		"Observaciones / Acción": "Cantidad de etapas estimuladas (stages).",
	},
	{
		"Variable Original": "tipo_terminacion",
		"Tipo Actual (Pandas)": str(df_datos_energia["tipo_terminacion"].dtype),
		"Nombre Propuesto": "tipo_terminacion",
		"Tipo Correspondiente": "Categórico nominal",
		"Dtype Sugerido": "category",
		"Valores Nulos": int(df_datos_energia["tipo_terminacion"].isnull().sum()),
		"Observaciones / Acción": "Método técnico de terminación (Plug & Perf, etc.).",
	},
	{
		"Variable Original": "arena_bombeada_nacional_tn",
		"Tipo Actual (Pandas)": str(df_datos_energia["arena_bombeada_nacional_tn"].dtype),
		"Nombre Propuesto": "arena_nacional_tn",
		"Tipo Correspondiente": "Numérico continuo",
		"Dtype Sugerido": "float64",
		"Valores Nulos": int(df_datos_energia["arena_bombeada_nacional_tn"].isnull().sum()),
		"Observaciones / Acción": "Agente de sostén local inyectado (toneladas).",
	},
	{
		"Variable Original": "arena_bombeada_importada_tn",
		"Tipo Actual (Pandas)": str(df_datos_energia["arena_bombeada_importada_tn"].dtype),
		"Nombre Propuesto": "arena_importada_tn",
		"Tipo Correspondiente": "Numérico continuo",
		"Dtype Sugerido": "float64",
		"Valores Nulos": int(df_datos_energia["arena_bombeada_importada_tn"].isnull().sum()),
		"Observaciones / Acción": "Agente de sostén importado inyectado (toneladas).",
	},
	{
		"Variable Original": "agua_inyectada_m3",
		"Tipo Actual (Pandas)": str(df_datos_energia["agua_inyectada_m3"].dtype),
		"Nombre Propuesto": "agua_inyectada_m3",
		"Tipo Correspondiente": "Numérico continuo",
		"Dtype Sugerido": "float64",
		"Valores Nulos": int(df_datos_energia["agua_inyectada_m3"].isnull().sum()),
		"Observaciones / Acción": "Volumen de fluido base agua en metros cúbicos.",
	},
	{
		"Variable Original": "co2_inyectado_m3",
		"Tipo Actual (Pandas)": str(df_datos_energia["co2_inyectado_m3"].dtype),
		"Nombre Propuesto": "co2_inyectado_m3",
		"Tipo Correspondiente": "Numérico continuo",
		"Dtype Sugerido": "float64",
		"Valores Nulos": int(df_datos_energia["co2_inyectado_m3"].isnull().sum()),
		"Observaciones / Acción": "Revisar si tiene varianza nula (solo ceros).",
	},
	{
		"Variable Original": "presion_maxima_psi",
		"Tipo Actual (Pandas)": str(df_datos_energia["presion_maxima_psi"].dtype),
		"Nombre Propuesto": "presion_maxima_psi",
		"Tipo Correspondiente": "Numérico continuo",
		"Dtype Sugerido": "float64",
		"Valores Nulos": int(df_datos_energia["presion_maxima_psi"].isnull().sum()),
		"Observaciones / Acción": "Presión pico de tratamiento en psi.",
	},
	{
		"Variable Original": "potencia_equipos_fractura_hp",
		"Tipo Actual (Pandas)": str(df_datos_energia["potencia_equipos_fractura_hp"].dtype),
		"Nombre Propuesto": "potencia_equipos_hp",
		"Tipo Correspondiente": "Numérico continuo",
		"Dtype Sugerido": "float64",
		"Valores Nulos": int(df_datos_energia["potencia_equipos_fractura_hp"].isnull().sum()),
		"Observaciones / Acción": "Potencia instalada en caballos de fuerza (51 nulos).",
	},
	{
		"Variable Original": "fecha_inicio_fractura",
		"Tipo Actual (Pandas)": str(df_datos_energia["fecha_inicio_fractura"].dtype),
		"Nombre Propuesto": "fecha_inicio_fractura",
		"Tipo Correspondiente": "Temporal",
		"Dtype Sugerido": "datetime64[ns]",
		"Valores Nulos": int(df_datos_energia["fecha_inicio_fractura"].isnull().sum()),
		"Observaciones / Acción": "Convertir con pd.to_datetime.",
	},
	{
		"Variable Original": "fecha_fin_fractura",
		"Tipo Actual (Pandas)": str(df_datos_energia["fecha_fin_fractura"].dtype),
		"Nombre Propuesto": "fecha_fin_fractura",
		"Tipo Correspondiente": "Temporal",
		"Dtype Sugerido": "datetime64[ns]",
		"Valores Nulos": int(df_datos_energia["fecha_fin_fractura"].isnull().sum()),
		"Observaciones / Acción": "Convertir con pd.to_datetime para medir duración.",
	},
	{
		"Variable Original": "fecha_data",
		"Tipo Actual (Pandas)": str(df_datos_energia["fecha_data"].dtype),
		"Nombre Propuesto": "fecha_registro",
		"Tipo Correspondiente": "Temporal",
		"Dtype Sugerido": "datetime64[ns]",
		"Valores Nulos": int(df_datos_energia["fecha_data"].isnull().sum()),
		"Observaciones / Acción": "Fecha de reporte al sistema.",
	},
	{
		"Variable Original": "anio_if",
		"Tipo Actual (Pandas)": str(df_datos_energia["anio_if"].dtype),
		"Nombre Propuesto": "anio_inicio_fractura",
		"Tipo Correspondiente": "Numérico / Ordinal",
		"Dtype Sugerido": "int32",
		"Valores Nulos": int(df_datos_energia["anio_if"].isnull().sum()),
		"Observaciones / Acción": "Redundante si se procesa fecha_inicio_fractura.",
	},
	{
		"Variable Original": "mes_if",
		"Tipo Actual (Pandas)": str(df_datos_energia["mes_if"].dtype),
		"Nombre Propuesto": "mes_inicio_fractura",
		"Tipo Correspondiente": "Categórico / Ordinal",
		"Dtype Sugerido": "int32",
		"Valores Nulos": int(df_datos_energia["mes_if"].isnull().sum()),
		"Observaciones / Acción": "Redundante con la columna temporal.",
	},
	{
		"Variable Original": "anio_ff",
		"Tipo Actual (Pandas)": str(df_datos_energia["anio_ff"].dtype),
		"Nombre Propuesto": "anio_fin_fractura",
		"Tipo Correspondiente": "Numérico / Ordinal",
		"Dtype Sugerido": "int32",
		"Valores Nulos": int(df_datos_energia["anio_ff"].isnull().sum()),
		"Observaciones / Acción": "Redundante si se procesa fecha_fin_fractura.",
	},
	{
		"Variable Original": "mes_ff",
		"Tipo Actual (Pandas)": str(df_datos_energia["mes_ff"].dtype),
		"Nombre Propuesto": "mes_fin_fractura",
		"Tipo Correspondiente": "Categórico / Ordinal",
		"Dtype Sugerido": "int32",
		"Valores Nulos": int(df_datos_energia["mes_ff"].isnull().sum()),
		"Observaciones / Acción": "Redundante con la columna temporal.",
	},
	{
		"Variable Original": "anio_carga",
		"Tipo Actual (Pandas)": str(df_datos_energia["anio_carga"].dtype),
		"Nombre Propuesto": "anio_carga_sistema",
		"Tipo Correspondiente": "Numérico / Ordinal",
		"Dtype Sugerido": "int32",
		"Valores Nulos": int(df_datos_energia["anio_carga"].isnull().sum()),
		"Observaciones / Acción": "Metadato administrativo de carga.",
	},
	{
		"Variable Original": "mes_carga",
		"Tipo Actual (Pandas)": str(df_datos_energia["mes_carga"].dtype),
		"Nombre Propuesto": "mes_carga_sistema",
		"Tipo Correspondiente": "Categórico / Ordinal",
		"Dtype Sugerido": "int32",
		"Valores Nulos": int(df_datos_energia["mes_carga"].isnull().sum()),
		"Observaciones / Acción": "Metadato administrativo de carga.",
	},
	{
		"Variable Original": "empresa_informante",
		"Tipo Actual (Pandas)": str(df_datos_energia["empresa_informante"].dtype),
		"Nombre Propuesto": "operadora",
		"Tipo Correspondiente": "Categórico nominal",
		"Dtype Sugerido": "category",
		"Valores Nulos": int(df_datos_energia["empresa_informante"].isnull().sum()),
		"Observaciones / Acción": "Compañía operadora del pozo.",
	},
	{
		"Variable Original": "mes",
		"Tipo Actual (Pandas)": str(df_datos_energia["mes"].dtype),
		"Nombre Propuesto": "mes_periodo",
		"Tipo Correspondiente": "Categórico / Ordinal",
		"Dtype Sugerido": "int32",
		"Valores Nulos": int(df_datos_energia["mes"].isnull().sum()),
		"Observaciones / Acción": "Posible columna redundante a contrastar.",
	},
	{
		"Variable Original": "anio",
		"Tipo Actual (Pandas)": str(df_datos_energia["anio"].dtype),
		"Nombre Propuesto": "anio_periodo",
		"Tipo Correspondiente": "Numérico / Ordinal",
		"Dtype Sugerido": "int32",
		"Valores Nulos": int(df_datos_energia["anio"].isnull().sum()),
		"Observaciones / Acción": "Posible columna redundante a contrastar.",
	},
]

# Crear el DataFrame de metadatos
df_diccionario = pd.DataFrame(mapeo_variables)

# Mostrar las 30 filas completas en el notebook
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

df_diccionario

In [ ]:
# @title Análisis de duplicados y cardinalidad de pozos
# Duplicados exactos en todo el DataFrame
print("Filas duplicadas exactas:", df_datos_energia.duplicated().sum())

# Duplicados en la clave primaria
print("IDs de fractura duplicados:", df_datos_energia.duplicated(subset=["id_base_fractura_adjiv"]).sum())

# Cuántas fracturas tiene cada pozo (cardinalidad de pozos)
print("Pozos únicos:", df_datos_energia["idpozo"].nunique())

In [ ]:
# @title Análisis de columnas con un único valor
# Ver columnas que tienen un único valor en todas sus filas
cols_univalor = [col for col in df_datos_energia.columns if df_datos_energia[col].nunique() == 1]
print("Columnas con un único valor (que no son útiles para modelar):", cols_univalor)

# Chequear específicamente co2_inyectado_m3 (suele ser 0 en la mayoría)
print("Distribución de CO2 inyectado:")
print(df_datos_energia["co2_inyectado_m3"].value_counts(dropna=False).head())

No se identificaron columnas con varianza estrictamente nula, por lo que ninguna variable queda descartada de forma automática por invariabilidad total.

El 96.8% de los registros (4.824 de 4.983 filas) presenta un valor de $0.0\text{ m}^3$, evidenciando que el uso de dióxido de carbono es una práctica marginal en las operaciones registradas.

**Debido a su comportamiento cuasi-constante, mantener esta variable en su escala continua introduce ruido sin aportar poder predictivo. Evaluar su eliminación para reducir dimensionalidad o transformarla en una variable dummy si el aspecto operativo resulta crítico para el objetivo**

In [ ]:
# @Cobertura temporal
# Convertir columnas a formato temporal

df_datos_energia["f_inicio"] = pd.to_datetime(df_datos_energia["fecha_inicio_fractura"], errors="raise")
df_datos_energia["f_fin"] = pd.to_datetime(df_datos_energia["fecha_fin_fractura"], errors="raise")
df_datos_energia["f_carga"] = pd.to_datetime(df_datos_energia["fecha_data"], errors="raise")

# 1. Rango global de operaciones
print(f"Inicio más antiguo: {df_datos_energia['f_inicio'].min().date()}")
print(f"Fin más reciente:   {df_datos_energia['f_fin'].max().date()}")

# 2. Duración de la operación (días de bombeo/estimulación)
df_datos_energia["duracion_dias"] = (df_datos_energia["f_fin"] - df_datos_energia["f_inicio"]).dt.total_seconds() / (24 * 3600)

print("\n--- Duración de Fractura (Días) ---")
print(df_datos_energia["duracion_dias"].describe(percentiles=[0.05, 0.5, 0.95]))

# 3. Latencia administrativa (cuánto tardan en cargar el dato al sistema desde que finaliza)
df_datos_energia["latencia_dias"] = (df_datos_energia["f_carga"] - df_datos_energia["f_fin"]).dt.total_seconds() / (24 * 3600)
print(f"\nMediana de tiempo de reporte administrativo: {df_datos_energia['latencia_dias'].median():.1f} días")

# 4. Volumen de operaciones por año
print("\n--- Operaciones por Año de Inicio ---")
print(df_datos_energia["f_inicio"].dt.year.value_counts().sort_index())

### Interpretación de la adquisición y la calidad

El archivo analizado contiene **4.910 registros y 30 variables**. La huella SHA-256 permite reconocer esta versión exacta del CSV, cuya fecha máxima de carga es el **29/09/2026**. Después de aplicar todos los criterios quedan **2.626 registros de 2.592 pozos**. La diferencia entre registros y pozos confirma que algunos pozos tienen más de una intervención; por eso `idpozo` se conserva como control y no como variable predictora.

Los filtros son secuenciales: `excluidos_en_paso` se interpreta respecto del paso anterior, no del archivo completo. Primero se retiraron 803 registros de otras cuencas y luego 376 correspondientes a reservorios convencionales dentro de la cuenca Neuquina. La mayor reducción posterior proviene de **1.071 registros sin una rama horizontal positiva**, coherente con el alcance limitado a pozos horizontales. También se excluyeron dos registros sin agua positiva y 32 con fecha de finalización anterior a la fecha de inicio.

El conjunto final representa aproximadamente el **53,5 %** del archivo original. Esta reducción no implica por sí sola mala calidad: una parte importante queda afuera por definición del alcance. En cambio, las fechas invertidas sí constituyen un problema de consistencia. Los resultados sólo describen el subconjunto retenido y no deben generalizarse a pozos convencionales, verticales u otras cuencas.